In [ ]:
import os
import zipfile
import ast
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from pydantic import BaseModel
# !pip install instructor
import instructor
from openai import OpenAI
from langchain_core.prompts import PromptTemplate

In [ ]:
import os

# Set the key as an environment variable

os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY_HERE"
# Initialize the openai client wrapped with Instructor
api_key = os.getenv("OPENAI_API_KEY")

In [ ]:
import os
import pandas as pd
import ast
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed


zip_path = "ablation_modified_bow.zip"
results_dir = "eeg_results_extracted_ablation_modified_bow"

# Ensure the directory exists and force extract the zip file
os.makedirs(results_dir, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(results_dir)

class Evaluation(BaseModel):
    fluency: int
    adequacy: int

api_key = os.getenv("OPENAI_API_KEY")
client = instructor.from_openai(OpenAI(api_key=api_key))

prompt = """You are a helpful language evaluator who can evaluate
input sentence2 and provide an evaluation of its fluency with a
likert scale rating of 1-5, 5 being highly fluent.
You will also have to compare two sentences and judge how adequate
is input sentence 2 with respect to input sentence 1, again with a likert scale rating of 1-5, 5 being highly adequate.
Here are the sentences: input_sentence1: {input_sentence1}, input_sentence2:{input_sentence2}"""

template = PromptTemplate(
    input_variables=["input_sentence1", "input_sentence2"],
    template=prompt,
)

def eval_single(index, row, client, template):
    """Refactored to take specific row data and return index for mapping"""
    try:
        input_sentence1 = row["gt_caption"]
        input_sentence2 = row["generated_caption"]

        final_prompt = template.format(input_sentence1=input_sentence1, input_sentence2=input_sentence2)

        eval_info = client.chat.completions.create(
            model="gpt-5-mini",
            response_model=Evaluation,
            messages=[{"role": "user", "content": final_prompt}],
        )
        return index, eval_info.model_dump()
    except Exception as e:
        print(f"Error at index {index}: {e}")
        return index, None

def get_results_for_single_file(results_directory, file_name, client, template):
    file_path = os.path.join(results_directory, file_name)
    output_filename = file_name.replace(".csv", "_eval.csv")
    output_path = os.path.join(results_directory, output_filename)
    
    # Load existing progress if it exists, otherwise load original
    if os.path.exists(output_path):
        df = pd.read_csv(output_path)
    else:
        df = pd.read_csv(file_path)
    
    # Ensure 'eval' column exists
    if 'eval' not in df.columns:
        df['eval'] = None

    # Identify rows that still need processing
    # We check for nulls or empty strings in the 'eval' column
    rows_to_process = df[df['eval'].isna() | (df['eval'] == "")]

    if rows_to_process.empty:
        print(f"All rows in {file_name} already processed.")
    else:
        num_threads = 32
        with ThreadPoolExecutor(max_workers=num_threads) as executor:
            # Only submit tasks for rows that don't have results yet
            futures = [
                executor.submit(eval_single, i, r, client, template) 
                for i, r in rows_to_process.iterrows()
            ]

            for future in tqdm(as_completed(futures), total=len(futures), desc=f"Rating {file_name}"):
                index, result = future.result()
                if result:
                    df.at[index, 'eval'] = str(result) # Store as string to keep CSV format clean
                    
                    # Optional: Save every 10 rows to minimize disk I/O while maintaining safety
                    if index % 10 == 0:
                        df.to_csv(output_path, index=False)
        
        # Final save for the file
        df.to_csv(output_path, index=False)

    # Calculate averages from the 'eval' column
    # Convert string back to dict for calculation
    eval_dicts = df['eval'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    avg_fluency = eval_dicts.apply(lambda x: x['fluency'] if x else 0).mean()
    avg_adequacy = eval_dicts.apply(lambda x: x['adequacy'] if x else 0).mean()

    return {"Avg. Fluency": avg_fluency, "Avg. Adequacy": avg_adequacy}


all_res = {}
output_summary_file = "final_gpt5_evaluations_ablation.csv"

# Process each file
for root, dirs, files in os.walk(results_dir):
    for file in files:
        # Only target original CSVs, not the ones we've already created (_eval.csv)
        if file.endswith(".csv") and not file.endswith("_eval.csv"):
            file_key = file.replace(".csv", "")
            
            print(f"\n--- Checking {file} ---")
            
            # Call the updated function with the required objects
            # It will now handle row-by-row skipping automatically
            results = get_results_for_single_file(root, file, client, template)
            
            # Store the averages in our main dictionary
            all_res[file_key] = results
            print(f"Current Averages for {file}: {results}")

            # Save the master summary file after every completed file
            summary_df = pd.DataFrame(all_res).transpose()
            summary_df.to_csv(output_summary_file)

if not all_res:
    print("\nWarning: No CSV files were found or processed. Check your file paths.")
else:
    print(f"\nEvaluation complete. All results saved to {output_summary_file}")